In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\AMAN\Downloads\qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [6]:
df.isnull().sum()

quote     0
Author    0
dtype: int64

In [7]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [8]:
quotes = quotes.str.lower()

In [9]:
# Removing Punctuations

In [10]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [11]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [13]:
tokinizer = Tokenizer()
tokinizer.fit_on_texts(quotes)

vocab_size = len(tokinizer.word_index) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 8979


In [14]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]    

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [15]:
sequence = tokinizer.texts_to_sequences(quotes)

In [16]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [17]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [18]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [19]:
len(X)

85271

In [20]:
# See X values they all are in not same size , So we use padding

In [21]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [23]:
# see X_padded and X for difference

In [24]:
y = np.array(y)

In [25]:
# one hot encoding

In [26]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [27]:
y_one_hot.shape

(85271, 8979)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [29]:
# Each word will be represented by a vector of length 50.
# The RNN will have 128 neurons (memory cells).

In [30]:
embedding_dim = 32
rnn_units = 64

In [31]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

C:\Users\AMAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [37]:
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
X_small = X_padded[:30000]
y_small = y[:30000]

In [ ]:
history = lstm_model.fit(
    X_small,
    y_small,
    epochs=3,
    batch_size=128,
    verbose=1
)

Epoch 1/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 62s 777ms/step - accuracy: 0.0413 - loss: 6.1227
Epoch 2/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 65s 818ms/step - accuracy: 0.0455 - loss: 5.9781
Epoch 3/3
47/79 ━━━━━━━━━━━━━━━━━━━━ 27s 856ms/step - accuracy: 0.0508 - loss: 5.8769

In [41]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 745, 32)             │         287,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 64)                  │          24,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8979)                │         583,635 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,687,387 (10.25 MB)

 Trainable params: 895,795 (3.42 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,791,592 (6.83 MB)

In [ ]:
# epochs =10
# batch_size = 128

history rnn = rnn_model.fit(
X_padded , y_one_hot,
epochs = epochs,
batch_size = batch_size,
validation_split = 0.1
)

In [42]:
lstm_model.save("quote_generator.keras")

In [44]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [45]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [46]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = int(np.argmax(pred))
  return index_to_word.get(pred_index, "")

In [47]:
seed_text = "The person"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

you


In [48]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if not next_word:
      break
    seed_text += " " + next_word
  return seed_text

In [49]:
seed_text = "the person"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

you


In [50]:
print("Vocabulary size:", len(tokinizer.word_index))
print("Model output size:", vocab_size)

Vocabulary size: 8978
Model output size: 8979


In [52]:
seed_text = "are you"

generated_text = generate_text(
    lstm_model,
    tokinizer,
    seed_text,
    max_len,
    15
)

print(generated_text)

are you you you you you you you you you you you you you you you you


In [ ]:
print(type(generate_text))

In [ ]:
print(history.history['loss'][-1])

In [ ]:
print("Quotes:", len(quotes))
print("Training samples:", len(X))
print("Vocabulary:", vocab_size)

In [ ]:
print(X_padded.shape)
print(y_one_hot.shape)